# Re-run the pipeline yourself

*A companion to `reproduce_paper.ipynb`.* That notebook replays every figure and
table in the paper from committed CSVs and JSON sidecars — it never touches the GP
emulator. This notebook does the opposite: we re-derive those sidecars from
scratch, one `(parameter, redshift, arm)` PySR fit at a time, so you can change a
prior, a fiducial value, the search budget, or the Sobolev weight and see where the
fit moves.

This is **Tier 3**: it needs the multi-fidelity GP emulator (`GPy` + the `lyaemu`
package) and a GP basedir (the trained flux vectors). Both live in one public
repo, `github.com/jibanCat/InferenceLyaData` (see the next cell). Every cell below
checks whether the emulator is available and, if not, prints the equivalent
command instead of failing, so the notebook always completes.

**Standing disclaimer.** A run from this notebook — quick or full — is run locally
by you, on your own budget, seed, and (optionally) your own fiducial or prior
tweaks. Treat every result here as illustrative. The paper's production run used a
far larger search budget (`niterations=200`, `populations=48`, a 5-seed band,
z in {2.6, 3.6, 4.2}); do not replace the paper's numbers with a rerun from this
notebook.

## 1. Get the GP emulator

The emulator has two parts, both in the public repo
[`github.com/jibanCat/InferenceLyaData`](https://github.com/jibanCat/InferenceLyaData):

- the **`lyaemu` package** (its `lyaemu/` directory) — the emulator code; put the
  clone root on `PYTHONPATH`;
- a **GP basedir** (its `Emulator_Files_KS/`) — the trained flux vectors the fits
  learn from; point `GP_BASEDIR` at it (or strip it to ~20 MB with
  `scripts/prep_kodiaq_gp.py`).

Set `LYA_EMULATOR` to your `InferenceLyaData` clone and `GP_BASEDIR` to the
basedir, then run the cell. (`GPy` + `emukit` must be pip-installed, with
`numpy < 2` for the GPy ABI.)

In [ ]:
import os, sys
from pathlib import Path
_root = Path.cwd()
if (_root / "src").is_dir(): sys.path.insert(0, str(_root / "src"))
# LYA_EMULATOR: a dir that CONTAINS the lyaemu/ package (an InferenceLyaData clone).
# GP_BASEDIR: a GP emulator basedir (InferenceLyaData/Emulator_Files_KS, or a
#             stripped data/kodiaq_gp).
LYA_EMULATOR = os.environ.get("LYA_EMULATOR", str(_root.parent / "InferenceLyaData"))
GP_BASEDIR = os.environ.get("GP_BASEDIR", "data/kodiaq_gp")
if LYA_EMULATOR and LYA_EMULATOR not in sys.path: sys.path.insert(0, LYA_EMULATOR)
try:
    import lyaemu; have_pkg = True
except Exception: have_pkg = False
try:
    import GPy; have_gpy = True
except Exception: have_gpy = False
have_gp = Path(GP_BASEDIR).is_dir()
EMULATOR_READY = have_pkg and have_gpy and have_gp
print("lyaemu package:", have_pkg, "| GPy:", have_gpy, "| GP basedir (" + GP_BASEDIR + "):", have_gp)
if not EMULATOR_READY:
    print(chr(10).join([
        "Emulator not ready. To provision (Tier 3) -- one public repo has BOTH the",
        "package and the data:",
        "  git clone https://github.com/jibanCat/InferenceLyaData ../InferenceLyaData",
        "  pip install GPy emukit          # numpy must stay <2 (GPy ABI)",
        "  export LYA_EMULATOR=$PWD/../InferenceLyaData         # its lyaemu/ is the package",
        "  export GP_BASEDIR=$LYA_EMULATOR/Emulator_Files_KS    # the KS GP basedir",
        "  #   (optional) strip the basedir to ~20 MB:",
        "  #   python scripts/prep_kodiaq_gp.py --source $GP_BASEDIR --dest data/kodiaq_gp",
        "then re-run this cell.",
    ]))

## 2. What the pipeline does

Every cell in the grid below is one PySR fit: fix a parameter (say `ns`), a
redshift `z`, and an arm (`value` or `sobolev`), and let PySR search for a
closed-form correction to `P1D(k; theta)` from the GP emulator's
one-parameter-varied (1pvar) sweep. Three steps happen for each grid cell:

1. **Normalize.** Flux/power from the GP and the parameter's own value are put on
   the same per-k, per-fidelity footing described in the paper's Normalization
   section, so an equation fit at one z or parameter transfers the same way
   everywhere else.
2. **Fit, then gate.** `refit_one_param_single_z` runs the PySR search that is the
   paper's per-parameter algorithm: the `value` arm trains on plain MSE; the
   `sobolev` arm adds a derivative-matching loss term of strength
   `sobolev_lambda`. `eval_grad_faithfulness.py` then scores every candidate on
   the Pareto front against the GP's own derivative and reads off the Pareto-knee
   row. A knee `grad_err <= 0.25` is the faithfulness gate used throughout the
   paper (Table 6).
3. **Combine.** A multi-parameter forecast adds the per-parameter 1D equations
   together — the paper's additive combine — rather than fitting one joint
   surface. `compare_to_production` (Section 7 below) and the multi-D figures in
   `reproduce_paper.ipynb` both check this combine against the GP directly.

`run_grid` loops arm x z x param over exactly this recipe and writes one
`grad_faith_<param>.csv` sidecar per cell into your run directory.

## 3. Configure your run

In [ ]:
from priya_forecast.rerun import RerunConfig, budget_warnings, cli_command_for
QUICK = True     # flip to False for the full 11 x {2.6,3.6,4.2} x {value,sobolev} run
cfg = RerunConfig.quick() if QUICK else RerunConfig.full()
cfg.basedir = GP_BASEDIR             # the GP basedir detected in cell 1
# --- knobs you can tweak (uncomment / edit) ---
# cfg.sobolev_lambda = 5.0          # derivative-matching strength
# cfg.maxsize = 20                   # equation complexity ceiling
# cfg.niterations = 30; cfg.populations = 8   # search budget (quick defaults)
# cfg.params = ["ns", "tau0", "Ap", "hub"]    # subset for speed
# cfg.seed = 0
# --- physics overrides: test your own hypothesis (Python-API only) ---
# cfg.fiducial_overrides = {"ns": 0.95}                 # move the fiducial n_s
# cfg.prior_overrides   = {"Ap": (1.0, 3.0)}            # widen the A_P prior
cfg.validate()
print("run dir:", cfg.run_dir)      # under results/tutorial_reruns/ (isolated, git-ignored)

## 4. Budget check

In [ ]:
w = budget_warnings(cfg)
if w:
    print("Differences from the production budget:")
    for line in w:
        print(" -", line)
else:
    print("This config matches the production grid and budget on every knob checked.")
print()
print("STANDING DISCLAIMER: any run from this notebook -- quick or full -- is run "
      "locally by you. Treat every result as illustrative. The paper's production "
      "numbers live under results/paper_production_20260630_perz_sobolev_z2.6-4.2/; "
      "do not replace them with a notebook rerun.")

## 5. Run

In [ ]:
from priya_forecast.rerun import run_grid
if EMULATOR_READY:
    run_dir = run_grid(cfg)
else:
    run_dir = cfg.run_dir
    print("Emulator not ready -- skipping the run. Equivalent CLI for one cell:")
    print(cli_command_for(cfg, cfg.params[0], cfg.zs[0], cfg.arms[0]))

## 6. Full run on a cluster

A `full()` config — 11 parameters x 3 redshifts x 2 arms, plus the paper's 5-seed
band and the ns budget-sensitivity control — is a SLURM job, not a notebook cell.
`scripts/submit_paper_production.sh` is the script that submitted the paper's
production run: `--dry-run` prints every `sbatch` command it would issue without
submitting anything.

In [ ]:
print("Full grid submission (SLURM, GreatLakes):")
print("  scripts/submit_paper_production.sh --dry-run   # print every sbatch command, submit nothing")
print("  scripts/submit_paper_production.sh              # submit for real")
print()
print("Per-cell CLI equivalents (one refit_one_param_single_z.py call each):")

seedband_cfg = RerunConfig.full()
seedband_cfg.seed = 1                       # one of the paper's 5 seed-band seeds (0-4)
print("--- seed-band, seed=1, z=3.6, sobolev ---")
print(cli_command_for(seedband_cfg, "ns", 3.6, "sobolev"))

sensitivity_cfg = RerunConfig.full()
sensitivity_cfg.maxsize = 35                # the ns budget-sensitivity control
print("--- ns budget-sensitivity, maxsize=35, z=3.6, value ---")
print(cli_command_for(sensitivity_cfg, "ns", 3.6, "value"))

## 7. How far did your run move? (vs production)

In [ ]:
from priya_forecast.rerun import compare_to_production
if EMULATOR_READY:
    cmp = compare_to_production(run_dir, zs=cfg.zs, arms=cfg.arms, params=cfg.params)
    import pandas as pd; pd.set_option("display.width", 160)
    print(cmp[["param","z","arm","grad_err_rerun","grad_err_prod","d_grad_err","flag","flipped"]].to_string(index=False))

## 8. Regenerate the paper figures from YOUR run

In [ ]:
import priya_forecast.paper_figures as pf
if EMULATOR_READY:
    run = pf.load_run(str(run_dir), z=cfg.zs[0])      # retarget the paper's plotting API
    display(pf.taxonomy(run))                          # the per-param taxonomy table
    pf.plot_scorecard(run)
    out = run_dir / "figures"; out.mkdir(exist_ok=True)
    pf.plot_pareto_faithfulness(run, out / "pareto_faithfulness.png")
    print("figures written under", out)

## 9. Knobs to try

A few concrete hypotheses, each a one-line change in Section 3 followed by a
rerun of Sections 5-8:

- **Raise the Sobolev weight.** `cfg.sobolev_lambda = 10.0` (production uses
  5.0). Expect the knee `grad_err` to drop further on borderline parameters like
  `ns`, at some cost to how well the plain value-space fit tracks the mean.
- **Try the DEFAULT operator set.** `cfg.smart_kwargs = False` swaps PySR's SMART
  set (`exp`, `log`, `square`, no trig) for the DEFAULT set (adds `sqrt`, `inv`,
  still no trig). Not strictly simpler — more operators to search over — but it
  changes which closed forms are reachable, and can move `grad_err` in either
  direction.
- **Move the fiducial.** `cfg.fiducial_overrides = {"tau0": <value>}` refits
  around a different fiducial optical depth. Rerun Section 7/8 and watch the
  taxonomy move — a parameter that is `faithful` at the paper's fiducial is not
  guaranteed to stay `faithful` elsewhere in the prior.
- **Widen a prior.** `cfg.prior_overrides = {"Ap": (0.5, 4.0)}` (paper uses
  (1.0, 3.0)). A wider range is a harder 1D regression problem; expect a
  shallower per-parameter slope and, often, a higher knee `grad_err`.

Every knob above is a config field or override — no code beyond Section 3 needs
to change.